In [ ]:
!pip install ortools

# Problema 1: Vehicle Routing

Una empresa de delivery debe asignar pedidos a sus vehículos de manera eficiente.
Se tienen varias ubicaciones que deben ser visitadas exactamente una vez.  
Cada vehículo inicia en un nodo específico y debe terminar en el depósito (nodo `0`).
Cada vehículo recorrerá una ruta visitando un subconjunto de nodos.

## Datos de entrada

- Un entero `N`: número total de nodos (incluyendo depósito `0`)
- Un entero `K`: número de vehículos
- Una matriz `D[N][N]` de distancias
- Un arreglo `starts[K]`: nodo inicial de cada vehículo
- Un entero `L`: distancia máxima permitida por vehículo

## Restricciones

1. Cada nodo `1..N-1` debe ser visitado exactamente una vez.

2. Cada vehículo:
   - inicia en `starts[k]`
   - termina en `0`

3. No se pueden repetir nodos.

4. La distancia total de cada ruta debe cumplir: `distancia ≤ L`

## Objetivo

Minimizar la **máxima distancia recorrida por cualquier vehículo**.

In [ ]:
import itertools
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp


def create_data():
    data = {}

    # data["distance_matrix"] = [
    #     [0, 2, 9, 10, 7],
    #     [2, 0, 6, 4, 3],
    #     [9, 6, 0, 8, 5],
    #     [10, 4, 8, 0, 6],
    #     [7, 3, 5, 6, 0],
    # ]
    # data["num_vehicles"] = 2
    # data["starts"] = [1, 2]
    # data["ends"] = [0, 0]
    # data["L"] = 20

    data["distance_matrix"] = [
        [0, 4, 8, 6, 7, 3, 9, 5, 6, 8, 9],
        [4, 0, 5, 3, 6, 4, 7, 6, 5, 7, 8],
        [8, 5, 0, 6, 4, 7, 3, 5, 4, 6, 7],
        [6, 3, 6, 0, 5, 4, 6, 4, 5, 6, 5],
        [7, 6, 4, 5, 0, 3, 5, 6, 4, 5, 6],
        [3, 4, 7, 4, 3, 0, 6, 5, 3, 4, 5],
        [9, 7, 3, 6, 5, 6, 0, 4, 5, 3, 4],
        [5, 6, 5, 4, 6, 5, 4, 0, 4, 5, 6],
        [6, 5, 4, 5, 4, 3, 5, 4, 0, 3, 4],
        [8, 7, 6, 6, 5, 4, 3, 5, 3, 9, 8],
        [9, 8, 7, 5, 6, 5, 4, 6, 4, 8, 0],
    ]

    data["num_vehicles"] = 3
    data["starts"] = [1, 2, 3]
    data["ends"] = [0, 0, 0]
    data["L"] = 35

    return data

### Método 1: Fuerza Bruta

In [ ]:
def brute_force_vrp(data):
    D = data["distance_matrix"]
    starts = data["starts"]
    L = data["L"]

    N = len(D)
    K = len(starts)
    nodes = list(range(1, N))

    def route_distance(route):
        return sum(D[route[i]][route[i + 1]] for i in range(len(route) - 1))

    best = float("inf")
    best_routes = None

    for assignment in itertools.product(range(K), repeat=len(nodes)):
        groups = [[] for _ in range(K)]

        for i, k in enumerate(assignment):
            groups[k].append(nodes[i])

        max_dist = 0
        valid = True
        routes = []

        for k in range(K):
            best_k = float("inf")
            best_route_k = None

            for perm in itertools.permutations(groups[k]):
                route = [starts[k]] + list(perm) + [0]
                dist = route_distance(route)

                if dist <= L and dist < best_k:
                    best_k = dist
                    best_route_k = route

            if best_route_k is None:
                valid = False
                break

            routes.append((best_route_k, best_k))
            max_dist = max(max_dist, best_k)

        if valid and max_dist < best:
            best = max_dist
            best_routes = routes

    return best, best_routes


data = create_data()
best, routes = brute_force_vrp(data)

print("Mejor max distancia:", best)

if routes:
    for i, (r, d) in enumerate(routes):
        print(f"Vehículo {i}: {r} -> {d}")

Mejor max distancia: 16
Vehículo 0: [1, 1, 2, 4, 5, 0] -> 15
Vehículo 1: [2, 6, 9, 7, 0] -> 16
Vehículo 2: [3, 3, 10, 8, 0] -> 15


### Método 2: Constraint Programming

In [ ]:
def print_solution(data, manager, routing, solution):
    max_route = 0
    routes = []

    for v in range(data["num_vehicles"]):
        index = routing.Start(v)
        route = []
        dist = 0

        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            route.append(node)
            prev = index
            index = solution.Value(routing.NextVar(index))
            dist += routing.GetArcCostForVehicle(prev, index, v)

        route.append(manager.IndexToNode(index))

        routes.append((route, dist))
        max_route = max(max_route, dist)

    print("Mejor max distancia:", max_route)
    for i, (r, d) in enumerate(routes):
        print(f"Vehículo {i}: {r} -> {d}")


def solve_cp(data):
    manager = pywrapcp.RoutingIndexManager(
        len(data["distance_matrix"]),
        data["num_vehicles"],
        data["starts"],
        data["ends"],
    )

    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        return data["distance_matrix"][manager.IndexToNode(from_index)][
            manager.IndexToNode(to_index)
        ]

    transit = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit)

    routing.AddDimension(
        transit,
        0,
        data["L"],
        True,
        "Distance",
    )

    dim = routing.GetDimensionOrDie("Distance")
    dim.SetGlobalSpanCostCoefficient(100)

    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )

    solution = routing.SolveWithParameters(params)

    if solution:
        print_solution(data, manager, routing, solution)


data = create_data()
solve_cp(data)

Mejor max distancia: 16
Vehículo 0: [1, 4, 5, 0] -> 12
Vehículo 1: [2, 6, 9, 7, 0] -> 16
Vehículo 2: [3, 10, 8, 0] -> 15


# Problema 2: Asignación Lineal

El problema de asignación lineal consiste en asignar un conjunto de tareas a un conjunto de agentes (trabajadores, máquinas, etc.) de manera que cada tarea se asigne a exactamente un agente y cada agente realice exactamente una tarea, minimizando el costo total.

### Ejemplo
Supongamos que tenemos 4 trabajadores y 4 tareas con la siguiente matriz de costos:

| | Tarea 0 | Tarea 1 | Tarea 2 | Tarea 3 |
| :--- | :---: | :---: | :---: | :---: |
| **Trabajador 0** | 90 | 80 | 75 | 70 |
| **Trabajador 1** | 35 | 85 | 55 | 65 |
| **Trabajador 2** | 125 | 95 | 90 | 105 |
| **Trabajador 3** | 45 | 110 | 95 | 115 |

In [ ]:
import itertools
from ortools.sat.python import cp_model

def create_assignment_data():
    """Prepara los datos siguiendo el formato del Ejercicio 1."""
    data = {}
    data['costs'] = [
        [90, 80, 75, 70],
        [35, 85, 55, 65],
        [125, 95, 90, 105],
        [45, 110, 95, 115],
    ]
    data['num_workers'] = len(data['costs'])
    data['num_tasks'] = len(data['costs'][0])
    return data

### Método 1: Fuerza Bruta


In [ ]:
def solve_assignment_brute_force(data):
    costs = data['costs']
    num_workers = data['num_workers']
    tasks = list(range(data['num_tasks']))

    min_cost = float('inf')
    best_assignment = None

    for p in itertools.permutations(tasks):
        current_cost = sum(costs[worker][task] for worker, task in enumerate(p))
        if current_cost < min_cost:
            min_cost = current_cost
            best_assignment = p

    return min_cost, best_assignment

data_assignment = create_assignment_data()
cost_fb, assign_fb = solve_assignment_brute_force(data_assignment)
print(f"[Fuerza Bruta] Costo mínimo: {cost_fb} | Asignación: {assign_fb}")

[Fuerza Bruta] Costo mínimo: 265 | Asignación: (3, 2, 1, 0)


### Método 2: Backtracking


In [ ]:
def solve_assignment_backtracking(data):
    costs = data['costs']
    num_workers = data['num_workers']
    num_tasks = data['num_tasks']

    best_cost = [float('inf')]
    assigned_tasks = [False] * num_tasks

    def backtrack(worker, current_total):
        if worker == num_workers:
            if current_total < best_cost[0]:
                best_cost[0] = current_total
            return

        if current_total >= best_cost[0]:
            return

        for task in range(num_tasks):
            if not assigned_tasks[task]:
                assigned_tasks[task] = True
                backtrack(worker + 1, current_total + costs[worker][task])
                assigned_tasks[task] = False

    backtrack(0, 0)
    return best_cost[0]

cost_bt = solve_assignment_backtracking(data_assignment)
print(f"[Backtracking] Costo mínimo: {cost_bt}")

[Backtracking] Costo mínimo: 265


### Método 3: Constraint Programming


In [ ]:
def solve_assignment_cp(data):
    model = cp_model.CpModel()
    costs = data['costs']
    num_workers = data['num_workers']
    num_tasks = data['num_tasks']

    x = {} # x[i, j] es 1 si worker i hace task j
    for i in range(num_workers):
        for j in range(num_tasks):
            x[i, j] = model.NewBoolVar(f'x_{i}_{j}')

    for i in range(num_workers):
        model.AddExactlyOne(x[i, j] for j in range(num_tasks))
    for j in range(num_tasks):
        model.AddExactlyOne(x[i, j] for i in range(num_workers))

    model.Minimize(sum(costs[i][j] * x[i, j] for i in range(num_workers) for j in range(num_tasks)))

    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    if status == cp_model.OPTIMAL:
        print(f'[CP-SAT] Costo total: {solver.ObjectiveValue()}')
        for i in range(num_workers):
            for j in range(num_tasks):
                if solver.Value(x[i, j]):
                    print(f'  Trabajador {i} -> Tarea {j} (Costo: {costs[i][j]})')

solve_assignment_cp(data_assignment)

[CP-SAT] Costo total: 265.0
  Trabajador 0 -> Tarea 3 (Costo: 70)
  Trabajador 1 -> Tarea 2 (Costo: 55)
  Trabajador 2 -> Tarea 1 (Costo: 95)
  Trabajador 3 -> Tarea 0 (Costo: 45)
